In [1]:
import os 
import pandas as pd
import networkx as nx 
import numpy as np
from tqdm.notebook import tqdm
import nx_parallel as nxp
import pickle
from scipy.stats import hypergeom

In [2]:
# read and process genesets for dictionary

genesets_path = "../data/gene_sets/c5.go.bp.v2026.1.Hs.symbols.gmt"
geneset_dict = {}

with open(genesets_path, 'r', encoding='UTF-8') as file:
    while line := file.readline():
        line = line.rstrip()
        line = line.split('\t')
        genes = line[2:-1]
        geneset_dict[line[0]] = genes


In [3]:
# read values for network

humannet = pd.read_csv("../data/raw/hnet_functional_symbols.tsv", sep="\t", header=None)
humannet.columns = ['source', 'target', 'score']
humannet.head(2)

hnet_nx = nx.from_pandas_edgelist(
    df=humannet,
    source='source',
    target='target'
)

seed_nodes = pd.read_csv("../cfg/seed_nodes_CP_sym.txt", header=None).values.flatten()

In [4]:
common_nodes = set(seed_nodes) & set(hnet_nx.nodes)
print(f"{len(common_nodes)} nodes in common, i.e.: {common_nodes}")
# build personalized
personalization = {
    seed: 1/len(common_nodes) for seed in common_nodes
}
k = len(common_nodes)
print(personalization)

nodes = [n for n in hnet_nx.nodes()] # nodes of the network

5 nodes in common, i.e.: {'PTPRD', 'STK38', 'KCTD20', 'PXT1', 'ETV7'}
{'PTPRD': 0.2, 'STK38': 0.2, 'KCTD20': 0.2, 'PXT1': 0.2, 'ETV7': 0.2}


In [2]:
# Pagerank with seed nodes
ppr_results = pd.read_csv("../output/ppr_results.tsv", sep="\t")
ppr_results.columns = ['Gene', 'PPR_score','zscore','rank']
ppr_results.head()

,Gene,PPR_score,zscore,rank
0,ETV7,0.030398,56.239347,1
1,PTPRD,0.030395,56.233595,2
2,STK38,0.030223,55.914846,3
3,KCTD20,0.030205,55.881102,4
4,PXT1,0.030033,55.562230,5


In [30]:
num_g = len(ppr_results[ppr_results['zscore'] >= 1.96]) # this is the number of the ppr geneset

# for each PPR permuted i select top num_g genes to build all the g_i 

In [18]:
# initialize random generator 
# run if output does not exists

if not os.path.exists("../output/pagerank_permutations.pkl"):

    rng = np.random.default_rng(seed=42)
    permutation_runs = []
    hnet_nx.__networkx_cache__.clear()

    for i in tqdm(range(1000)):
        
        samples = rng.choice(nodes, size=k, replace=False)
        personalization_permut = {
            seed: 1/k for seed in samples
        }
        ppr = nx.pagerank(
            G=hnet_nx,
            personalization=personalization_permut,
            tol=1e-05,
            backend="graphblas"
        )
        ordered_values =[ppr[n] for n in nodes]
        permutation_runs.append(ordered_values)

    hnet_nx.__networkx_cache__.clear()

    with open("../output/pagerank_permutations.pkl","wb") as out_file:
        pickle.dump(permutation_runs, out_file)

  0%|          | 0/1000 [00:00<?, ?it/s]

c:\Users\simone\anaconda3\envs\post-gwas\Lib\site-packages\networkx\utils\backends.py:1386: UserWarning: Note: conversions to backend graphs are saved to cache (`G.__networkx_cache__` on the original graph) by default.

This warning means the cached graph is being used for the 'graphblas' backend in the call to pagerank.

For the cache to be consistent (i.e., correct), the input graph must not have been manually mutated since the cached graph was created. Examples of manually mutating the graph data structures resulting in an inconsistent cache include:

    >>> G[u][v][key] = val

and

    >>> for u, v, d in G.edges(data=True):
    ...     d[key] = val

Using methods such as `G.add_edge(u, v, weight=val)` will correctly clear the cache to keep it consistent. You may also use `G.__networkx_cache__.clear()` to manually clear the cache, or set `G.__networkx_cache__` to None to disable caching for G. Enable or disable caching globally via `nx.config.cache_converted_graphs` config.

To dis

In [23]:
with open("../output/pagerank_permutations.pkl","rb") as in_file:
    permutation_runs= pickle.load(in_file)

In [ ]:
permutation_runs = np.array(permutation_runs)
print(permutation_runs.shape)

top_permut_idx = np.argpartition(permutation_runs*-1, kth=num_g, axis=1)[:,:num_g]

(1000, 18459)


In [ ]:
# top genes for all permutations
g_permut = np.array(nodes, dtype=str)[top_permut_idx]

# names for all gene sets
gene_set_names = list(geneset_dict.keys())

In [94]:
def enrichment_score(
        geneset,
        ref_geneset,
        background
):
        geneset = set(geneset)
        ref_geneset = set(ref_geneset)

        background_size = len(background)
        gene_ratio = len(geneset & ref_geneset)/len(geneset)
        bg_ratio = len(ref_gi)/background_size
        fold_enrich = gene_ratio/bg_ratio

        pval = hypergeom.sf(len(geneset & ref_geneset) - 1, background_size, len(geneset), len(ref_geneset))

        return [fold_enrich, pval]

In [ ]:
# calculate observed T statistic

geneset = ppr_results[ppr_results['zscore'] >= 1.68]['Gene'].to_list()

n_permut = 1000 

In [96]:
# calculate empirical nominal pvalues

nominal_scores = {}
nominal_pval = {}

for k,v in tqdm(geneset_dict.items()):
    T_permut_l = []

    nominal_scores[k] = enrichment_score(geneset, v, nodes)[0]

    for i in range(n_permut):
        in_gi = set(g_permut[i])
        T_permut_l.append(enrichment_score(in_gi, v, nodes)[0])

    nominal_pval[k] = (1+np.sum(np.array(T_permut_l)>=nominal_scores[k]))/(n_permut+1)

  0%|          | 0/7538 [00:00<?, ?it/s]

In [101]:
# MaxT correction

# Calculate null T distribution

# pval empirico 
# pemp_nominal = (1 + #{T_permut >= T_obs})/(num_permut  +1)
# pemp_corrected = (1 + #{T_max_permut >= T_obs})/(num_permut  +1))
# Dudoit, S., Shaffer, J.P., and Boldrick, J.C. (2003). Multiple Hypothesis Testing in Microarray Experiments. Statistical Science, 18(1), 71–103.

background_size = len(nodes)
T_max_l = []

for i in tqdm(range(n_permut)):
    in_gi = set(g_permut[i])
    T_max = 0
    
    for k,v in geneset_dict.items():
       
        fold_enrich = enrichment_score(in_gi,v,nodes)[0]
        
        if fold_enrich >= T_max: 
            T_max = fold_enrich

    T_max_l.append(T_max)


  0%|          | 0/1000 [00:00<?, ?it/s]

In [102]:
corrected_pvals = {}

for k,v in tqdm(geneset_dict.items()):
    
    observed_score = nominal_scores[k]
    corrected_pvals[k] = (1+np.sum(np.array(T_max_l)>=observed_score))/(n_permut+1)


  0%|          | 0/7538 [00:00<?, ?it/s]

In [ ]:
enrichment_scores_df = (
    pd.DataFrame.from_dict(nominal_scores, orient='index', columns=["Nom-scores"])
    .merge(
        pd.DataFrame.from_dict(nominal_pval, orient='index', columns=["Nom-pval"]),
        left_index=True,
        right_index=True,
    )
    .merge(
        pd.DataFrame.from_dict(corrected_pvals, orient='index', columns=["MaxT-pval"]),
        left_index=True,
        right_index=True,
    )
)


In [3]:
geneset = ppr_results[ppr_results['zscore'] >= 1.68]['Gene'].to_list()
for g in geneset:
    print(g)

ETV7
PTPRD
STK38
KCTD20
PXT1
MEOX2
TRIM25
XPO1
CUL3
DUSP16
PPP1CA
MARK4
PPP1CC
BUB3
ZC3HAV1
COPS7A
MRFAP1L1
UTP14C
UGP2
KIAA0232
BTBD10
SOX13
SPI1
ELK1
SPIB
ELK4
ERG
ELK3
SPIC
ETV6
ETV2
FEV
SPDEF
ETS1
ELF3
GABPA
ETS2
FLI1
EHF
ELF5
RICTOR
RPTOR
PDLIM7
VTN
BATF2
L3MBTL1
IRF4


In [120]:
enrichment_scores_df.sort_values('Nom-pval', ascending=True).head(10)

,Nom-scores,Nom-pval,MaxT-pval
GOBP_BLASTOCYST_DEVELOPMENT,212.988462,0.000999,0.996004
GOBP_COP9_SIGNALOSOME_ASSEMBLY,35.498077,0.001998,1.000000
GOBP_ECTODERMAL_CELL_DIFFERENTIATION,70.996154,0.001998,1.000000
GOBP_IMMATURE_B_CELL_DIFFERENTIATION,70.996154,0.003996,1.000000
GOBP_ECTODERM_DEVELOPMENT,70.996154,0.004995,1.000000
GOBP_REGULATION_OF_MESODERM_DEVELOPMENT,35.498077,0.004995,1.000000
GOBP_BLASTOCYST_FORMATION,70.996154,0.006993,1.000000
GOBP_POSITIVE_REGULATION_OF_TRANSCRIPTION_BY_RNA_POLYMERASE_II,709.961538,0.006993,0.183816
GOBP_CELL_SURFACE_RECEPTOR_PROTEIN_TYROSINE_PHOSPHATASE_SIGNALING_PATHWAY,35.498077,0.007992,1.000000
GOBP_NEGATIVE_REGULATION_OF_MHC_CLASS_II_BIOSYNTHETIC_PROCESS,35.498077,0.007992,1.000000
